In [1]:
import cv2
import numpy as np

In [2]:
src = cv2.imread("./Data/example.png")

# 그레이스케일로 변환
img_gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# 이진화 (임계값 127, 최대값 255, 이진화 타입)
# 127 기준 작으면 0, 크면 255
retval, img_thresh = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)

print(retval)

cv2.imshow("src", src)
cv2.imshow("img_gray", img_gray)
cv2.imshow("img_thresh", img_thresh)

cv2.waitKey(0)
cv2.destroyAllWindows()

127.0


In [3]:
# 픽셀값이 임계값보다 크면 최대값, 그렇지 않으면 0으로 설정
_, img_thresh_bin = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)

# 픽셀값이 임계값보다 크면 0, 그렇지 않으면 최대값으로 설정
_, img_thresh_inv = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY_INV)

# 픽셀값이 임계값보다 크면 임계값으로 설정, 그렇지 않으면 원래값 유지
_, img_thresh_trunc = cv2.threshold(img_gray, 127, 255, cv2.THRESH_TRUNC)

# 픽셀값이 임계값보다 크면 원래값 유지, 그렇지 않으면 0으로 설정
_, img_thresh_tozero = cv2.threshold(img_gray, 127, 255, cv2.THRESH_TOZERO)

# 픽셀값이 임계값보다 크면 0으로 설정, 그렇지 않으면 원래값 유지
_, img_thresh_tozero_inv = cv2.threshold(img_gray, 127, 255, cv2.THRESH_TOZERO_INV)

cv2.imshow("img_gray", img_gray)
cv2.imshow("img_thresh_bin", img_thresh_bin)
cv2.imshow("img_thresh_inv", img_thresh_inv)
cv2.imshow("img_thresh_trunc", img_thresh_trunc)
cv2.imshow("img_thresh_tozero", img_thresh_tozero)
cv2.imshow("img_thresh_tozero_inv", img_thresh_tozero_inv)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [2]:
# 글씨에 적용
src = cv2.imread("./Data/godard.png")

img_gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

retval, img_thresh = cv2.threshold(img_gray, 240, 255, cv2.THRESH_BINARY)

print(retval)

cv2.imshow("src", src)
cv2.imshow("img_gray", img_gray)
cv2.imshow("img_thresh", img_thresh)

cv2.waitKey(0)
cv2.destroyAllWindows()

240.0


In [3]:
# 트랙바 사용하여 적절한 Threshold 찾기
def on_trackbar(val):
    threshold = val
    _, binary_image = cv2.threshold(gray_image, threshold, 255, cv2.THRESH_BINARY)
    cv2.imshow('Binary Image', binary_image)

gray_image = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

cv2.namedWindow('Binary Image')
cv2.createTrackbar('Threshold', 'Binary Image', 127, 255, on_trackbar)

_, binary_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)
cv2.imshow('Binary Image', binary_image)

while True:
    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC 눌러서 종료
        break

cv2.destroyAllWindows()

In [4]:
# Otsu의 이진화 알고리즘: 이미지의 히스토그램을 분석하여 최적의 임계값을 찾아낸다.
# 1. 히스토그램 계산: 이미지의 히스토그램(각 픽셀값의 빈도)을 계산
# 2. 클래스 분리: 가능한 모든 임계값을 기준으로 이미지를 두 클래스(밝은 부분, 어두운 부분)로 분리
# 3. 분산 계산: 각 클래스의 분산을 계산하고 두 클래스 간의 분산이 최대가 되는 임계값을 찾는다.
_, img_thresh = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)
_, img_thresh_otsu = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, img_thresh_otsu_inv = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

cv2.imshow("src", src)
cv2.imshow("img_gray", img_gray)
cv2.imshow("img_thresh", img_thresh)
cv2.imshow("img_thresh_otsu", img_thresh_otsu)
cv2.imshow("img_thresh_otsu_inv", img_thresh_otsu_inv)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
# 이미지 모폴로지 연산 (침식, 팽창, 열기, 닫기, 모폴로지 그레디언트)
src = np.full((500, 500), 0, dtype=np.uint8)

cv2.putText(src, "HAM", (50, 300), cv2.FONT_HERSHEY_SIMPLEX, 6, 255, 20, cv2.LINE_AA)

noise = np.random.randint(0, 2, (500, 500)).astype(np.uint8) * 255

img_dot = cv2.bitwise_or(src, noise)
img_hole = cv2.bitwise_and(src, noise)

cv2.imshow("src", src)
cv2.imshow("img_dot", img_dot)
cv2.imshow("img_hole", img_hole)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [6]:
# 구조 요소 정의 (5x5 크기의 정사각형 커널)
kernel = np.ones((5, 5), np.uint8)

# 모폴로지 연산 적용
erosion = cv2.erode(img_dot, kernel, iterations=1) # 침식
dilation = cv2.dilate(img_hole, kernel, iterations=1) # 팽창
opening = cv2.morphologyEx(img_dot, cv2.MORPH_OPEN, kernel) # 열기: 침식 후 팽창
closing = cv2.morphologyEx(img_hole, cv2.MORPH_CLOSE, kernel) # 닫기: 팽창 후 침식

cv2.imshow("img_dot", img_dot)
cv2.imshow("img_hole", img_hole)
cv2.imshow("erosion", erosion)
cv2.imshow("dilation", dilation)
cv2.imshow("opening", opening)
cv2.imshow("closing", closing)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [7]:
# 침식 4번
erosion4 = cv2.erode(img_dot, kernel, iterations=4)

cv2.imshow("img_dot", img_dot)
cv2.imshow("erosion", erosion)
cv2.imshow("erosion4", erosion4)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [8]:
# dot 그림에 팽창 사용
dilation_dot = cv2.dilate(img_dot, kernel, iterations=1)

cv2.imshow("img_dot", img_dot)
cv2.imshow("erosion", erosion)
cv2.imshow("dilation_dot", dilation_dot)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
# 모폴로지 그레디언트: 같은 그림에 대한 침식, 팽창 차이를 한번에 추출
gradient_dot = cv2.morphologyEx(img_dot, cv2.MORPH_GRADIENT, kernel)
cv2.imshow("img_dot", img_dot)
cv2.imshow("erosion", erosion)
cv2.imshow("dilation_dot", dilation_dot)
cv2.imshow("gradient_dot", gradient_dot)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
gradient_hole = cv2.morphologyEx(img_hole, cv2.MORPH_GRADIENT, kernel)
erosion_hole = cv2.erode(img_hole, kernel, iterations=1)

cv2.imshow("img_hole", img_hole)
cv2.imshow("erosion_hole", erosion_hole)
cv2.imshow("dilation", dilation)
cv2.imshow("gradient_hole", gradient_hole)
cv2.waitKey(0)
cv2.destroyAllWindows()